# Module 38.3 — Augmentation Strategy

**Jupyter Notebook Notes · Computer Vision · AI/ML Engineer Roadmap**

---

## 1. What is Data Augmentation?

Data augmentation is a technique used to create modified versions of existing training images by applying transformations such as rotation, cropping, flipping, brightness adjustment, and noise addition.

The objective is to help a model learn useful visual features rather than memorising the exact appearance of its training images.

### Practical Example

Suppose you have 500 images of defective motor components. Instead of relying only on those original appearances, you can introduce realistic variations in lighting, orientation, and image framing.

![Data augmentation example](https://images.openai.com/static-rsc-4/qLmLiCvwdnTAWiLbywApNM2RAo98dPjYTrASkfa7bSyrEwdcioSsz2qNbmqJCVDfuoUUPRJQVN5obx2GBG1j41meMi8Bckz1wfvB3BULdRxgWLi9Dhtoh5LVwUnIJVAbFCVCX3qnsxzFYeJYeqZt6av__Y-1FxHWsGNr5BXzGmk?purpose=inline)

### Why Do We Need Augmentation?

Without sufficient variation, a model may:

- Memorise training images.
- Become sensitive to small changes in image appearance.
- Perform poorly on images captured under different lighting conditions.
- Fail to generalise to unseen environments.

Augmentation helps improve generalisation and robustness.

> **Important distinction:** Augmentation does not collect new real-world information. It creates transformed training examples from existing data.

---

## 2. Three Augmentation Strategies

We will compare three approaches:

1. No augmentation.
2. Mild augmentation.
3. Aggressive augmentation.

### A. No Augmentation

No augmentation means the model receives the original training images without random transformations.

Basic preprocessing such as resizing, tensor conversion, and normalization can still be applied.

**Example:**

```python
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
```

### What Happens?

Suppose the training dataset contains 1,000 images.

The model repeatedly sees those same 1,000 images across training epochs, without random augmentation.

### Advantages

- Simple training pipeline.
- Original image characteristics are preserved.
- Useful as a baseline for experiments.
- No risk of introducing unrealistic transformations through augmentation.

### Disadvantages

- Limited visual diversity.
- Greater risk of overfitting, especially with small datasets.
- May perform poorly when real-world image conditions change.

### When to Use It

- As a baseline experiment.
- When you have a large, diverse dataset.
- When transformations could change the meaning of the image.

---

### B. Mild Augmentation

Mild augmentation applies small, realistic changes to training images.

Examples:

- Small rotations.
- Horizontal flipping.
- Slight brightness changes.
- Moderate cropping.
- Small contrast adjustments.

![Mild data augmentation](https://images.openai.com/static-rsc-4/AEz2o-PtQhGNceI21GnAhcnYPwMoTV2LAr1k8BON0Du8AdzvxsdGfNFnWe-0MVcwxT0w-LVdO4QumKL5OWptjqbjYo9bnSiLSgUwc1Teum85aZRSSkh_ZdPyuT1-leCOsAiB-4jx9GMuBB275PqsNkCxbhxMb_zva1CNUpOgZHw?purpose=inline)

### Example Using PyTorch

```python
from torchvision import transforms

mild_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor()
])
```

### Understanding the Parameters

| Parameter | Meaning |
|---|---|
| `p=0.5` | 50% probability of horizontal flipping |
| `degrees=10` | Random rotation within approximately ±10° |
| `brightness=0.15` | Small brightness variation |
| `contrast=0.15` | Small contrast variation |

The model encounters different but realistic versions of the same training image.

### Advantages

- Introduces useful visual diversity.
- Can reduce overfitting.
- Helps improve robustness to common image variations.
- Usually a sensible starting point for small and medium-sized datasets.

### Disadvantages

- May not be sufficient for highly variable environments.
- Incorrect transformations can still affect image meaning.

### When to Use It

- Small datasets.
- Standard image classification.
- Manufacturing inspection with limited camera variation.
- When real-world images have moderate lighting and orientation differences.

---

### C. Aggressive Augmentation

Aggressive augmentation applies stronger transformations or combines multiple transformations to create substantially different training images.

Examples:

- Large rotations.
- Strong colour distortion.
- Random resized cropping.
- Gaussian noise.
- Blur.
- Random erasing.
- MixUp and CutMix.

![Aggressive data augmentation](https://images.openai.com/static-rsc-4/LskICj6wESgKKVhZzOSjcYezH2G-8wYxOf7o6o2xRGEFtipPC1IbOTuezANufqzVDMAiIRfzl8-TQkrQNyCbm6rl89eBuGJTd8V9oKL1cCABNKhUqhSYLI4_0eIpJt20npph8y8mXFQtJ1ftjxAmvGzg1dGeJcHP2UWjdCtUaDU?purpose=inline)

### Example Using PyTorch

```python
from torchvision import transforms

aggressive_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.5, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=40),
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5,
        hue=0.1
    ),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5)
])
```

### What Is Happening Here?

| Transformation | Effect |
|---|---|
| RandomResizedCrop | Crops a random portion and resizes it |
| RandomRotation | Introduces larger orientation changes |
| ColorJitter | Changes colour and lighting properties |
| RandomGrayscale | Sometimes removes colour information |
| RandomErasing | Masks a random region of the tensor |

### Advantages

- Can improve robustness under substantial visual variation.
- Useful when training data is limited and transformations are realistic.
- Can help prevent the model from depending on small visual details.

### Disadvantages

- Can destroy important visual information.
- May generate unrealistic training samples.
- Can make learning more difficult.
- Excessive transformations can reduce classification performance.

### When to Use It

- Highly variable image environments.
- Large differences in camera conditions.
- Experiments where mild augmentation is insufficient.
- Carefully validated robustness training.

> **Important:** Aggressive augmentation does not automatically produce a better model. More augmentation is not always better.

---

## 3. Direct Comparison

| Feature | No Augmentation | Mild Augmentation | Aggressive Augmentation |
|---|---|---|---|
| Image variation | None beyond preprocessing | Small | Large |
| Training diversity | Low | Moderate | High |
| Overfitting risk | Potentially higher | Often reduced | Can be reduced, but not guaranteed |
| Training difficulty | Lower | Moderate | Potentially higher |
| Risk of unrealistic images | Very low | Low to moderate | Higher |
| Recommended starting point | Baseline | General-purpose experiments | Specialised experiments |

### Visual Understanding

Think of it this way:

- **No augmentation:** Learn from the original photographs.
- **Mild augmentation:** Learn from realistic variations of the photographs.
- **Aggressive augmentation:** Learn from much more varied transformations, some of which may be unrealistic.

---

## 4. How Augmentation Affects Overfitting

Connect this to the overfitting concepts you already studied in Deep Learning.

Consider two observations:

### Scenario 1 — No Augmentation

- Training accuracy: 99%
- Validation accuracy: 82%

The large gap may indicate overfitting.

### Scenario 2 — Mild Augmentation

- Training accuracy: 94%
- Validation accuracy: 90%

The model may have learned more generalisable features.

![Model generalisation](https://images.openai.com/static-rsc-4/MFWfmntqbmEOEk2TI7Qe-aA2l3lYU3_50P7170WyTQ7XHtIG5EpQReuNXta1gsHrCtXbHOA41GiGoK7gPWFW59B8WpCHUpOk3MqfkKXontbEs3le1qv5zRKRsrBRUgWbxX0vpFt2ikWrhY2eVPlCOvYjXyt3uh3lZ447Q_FKd8k?purpose=inline)

These numbers are illustrative, not guaranteed outcomes.

### Why Might Training Accuracy Decrease?

With augmentation, the model receives more challenging variations of the original images.

Therefore, training accuracy may decrease while validation performance improves.

**Key idea:** We are not necessarily trying to maximise training accuracy. We want good performance on unseen real-world images.

---

## 5. Augmentation Must Preserve the Label

This is one of the most important engineering principles.

A transformation is useful only when it preserves the correct class label.

Consider a motor defect classification dataset.

Suppose the original image is labelled:

```text
Class: Crack
```

### Safe Transformations Might Include

- Slight brightness changes.
- Small rotations.
- Mild contrast changes.
- Realistic camera noise.

### Potentially Unsafe Transformations

- Cropping out the crack completely.
- Applying strong blur that hides the crack.
- Excessive rotation if orientation has meaning.
- Strong colour changes that alter the appearance of corrosion.

The transformed image must still represent the same class.

### Engineering Rule

> Never apply an augmentation simply because a library provides it. Apply it because it represents a variation that could realistically occur in your application.

---

## 6. Training vs. Validation and Test Augmentation

This distinction is critical.

| Dataset | Augmentation Strategy |
|---|---|
| Training | Random augmentation is generally applied |
| Validation | Consistent deterministic preprocessing |
| Testing | Consistent deterministic preprocessing |

### Why?

We want to train on varied examples but evaluate on a consistent representation of real-world images.

### Example

```python
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
```

For pretrained models, add the normalization expected by their weights to both training and evaluation pipelines.

---

## 7. Comparing All Three Strategies Experimentally

In an actual AI/ML project, do not decide which strategy is best based only on intuition.

Run controlled experiments.

| Experiment | Training Transformation |
|---|---|
| Experiment 1 | No augmentation |
| Experiment 2 | Mild augmentation |
| Experiment 3 | Aggressive augmentation |

### Keep the Following Constant

- Dataset splits.
- Model architecture.
- Optimizer.
- Learning rate.
- Batch size.
- Training epochs.
- Evaluation procedure.

Then compare validation performance.

### Example Experimental Results

<text> </text>

| Metric | No Augmentation | Mild | Aggressive |
|---|---:|---:|---:|
| Training accuracy | 99% | 95% | 88% |
| Validation accuracy | 83% | 91% | 85% |
| Validation loss | 0.62 | 0.31 | 0.49 |

*Illustrative results only; actual performance depends on the dataset and model.*

In this example, mild augmentation performs best.

### Why Might Aggressive Augmentation Perform Worse?

Because some transformations may have removed or distorted features needed for classification.

---

## 8. What You Should Remember 🧠

1. Augmentation creates modified training examples from existing images.
2. No augmentation is useful as a baseline.
3. Mild augmentation introduces realistic variation and is a good starting point.
4. Aggressive augmentation can improve robustness, but excessive transformations may harm learning.
5. Augmentation must preserve the original class label.
6. Apply random augmentation to training data, not normally to validation or test data.
7. Compare strategies through controlled experiments rather than assuming aggressive augmentation is better.

---

